In [ ]:
# Only use when connecting to Colab GPU from VSCode extension
# Sets up Colab environment --> Should take ~3 minutes

from google.colab import drive
import os
drive.mount('/content/drive')

!mkdir -p ~/.ssh
!cp /content/drive/MyDrive/colab_keys/id_ed25519 ~/.ssh/
!cp /content/drive/MyDrive/colab_keys/id_ed25519.pub ~/.ssh/

!chmod 600 ~/.ssh/id_ed25519
!ssh-keyscan github.com >> ~/.ssh/known_hosts

if not os.path.exists("CMPM118-Winter2026"):
    !git clone git@github.com:Tardigrades3/CMPM118-Winter2026.git
    
%cd CMPM118-Winter2026
!git checkout aran-dev
# !pip install -e .
!python download_ninapro.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
# github.com:22 SSH-2.0-a73f77f
# github.com:22 SSH-2.0-a73f77f
# github.com:22 SSH-2.0-a73f77f
# github.com:22 SSH-2.0-a73f77f
# github.com:22 SSH-2.0-a73f77f
Cloning into 'CMPM118-Winter2026'...
remote: Enumerating objects: 173, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 173 (delta 10), reused 21 (delta 7), pack-reused 147 (from 2)
Receiving objects: 100% (173/173), 608.16 MiB | 7.26 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/CMPM118-Winter2026/CMPM118-Winter2026
Branch 'aran-dev' set up to track remote branch 'aran-dev' from 'origin'.
Switched to a new branch 'aran-dev'
Obtaining file:///content/CMPM118-Winter2026/CMPM118-Winter2026
ERROR: file:///content/CMPM118-Winter2026/CMPM118-Winter2026 does not appear to be a Python project: neither 'setup.py' nor 'pyproject.tom

In [17]:
import torch
from torch import nn
from tqdm import tqdm
import copy
from torch.utils.data import Subset
import torch.nn.functional as F

import sys, os
sys.path.append(os.path.abspath("../"))
import preprocessing



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

# Base MNIST NN

class NinaProClassificationModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv1d(10, 64, kernel_size=5, stride=2)
    self.relu1 = nn.ReLU()
    self.pool1 = nn.MaxPool1d(kernel_size=2)

    self.conv2 = nn.Conv1d(64, 128, kernel_size=5, stride=2)
    self.relu2 = nn.ReLU()
    self.pool2 = nn.MaxPool1d(kernel_size=2)

    self.flatten = nn.Flatten()
    
    self.dense1 = nn.LazyLinear(256)
    self.relu3 = nn.ReLU()
    self.dense2 = nn.Linear(256, 128)
    self.relu4 = nn.ReLU()
    self.dense3 = nn.Linear(128, 13)

  def forward(self, x):
    out = self.relu1(self.conv1(x))
    out = self.pool1(out)
    out = self.relu2(self.conv2(out))
    out = self.pool2(out)
    out = self.flatten(out)
    out = self.relu3(self.dense1(out))
    out = self.relu4(self.dense2(out))
    out = self.dense3(out)
    return out

class NinaProDataset(torch.utils.data.Dataset):
  def __init__(self, x, y):
    self.X = torch.tensor(x, dtype=torch.float32)
    self.y = torch.tensor(y, dtype=torch.long)

  def __len__(self):
    return len(self.X)
  
  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

# Base NN


model = NinaProClassificationModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# path should be ../NinaProData for local machine
x_train, y_train, x_test, y_test, class_weights = preprocessing.multi_preprocess(exercise_number=1, path="./NinaProData")
num_batches = 100

train_data = NinaProDataset(x_train, y_train)
test_data = NinaProDataset(x_test, y_test)
batched_train = torch.utils.data.DataLoader(train_data, batch_size=num_batches, shuffle=True)
batched_test = torch.utils.data.DataLoader(test_data, batch_size=num_batches, shuffle=False)

loss_func = nn.CrossEntropyLoss()

Using cuda device
subject #[[1]]
exercise #[[1]]
0.0     351
5.0      58
3.0      53
1.0      50
8.0      44
6.0      43
12.0     42
4.0      41
10.0     39
2.0      37
7.0      36
9.0      35
11.0     35
Name: count, dtype: int64
0.0     908
3.0      30
1.0      26
10.0     25
12.0     23
5.0      22
4.0      18
9.0      17
6.0      17
7.0      17
2.0      16
8.0      16
11.0     15
Name: count, dtype: int64
subject #[[2]]
exercise #[[1]]
0.0     291
5.0      55
7.0      54
3.0      53
8.0      50
11.0     50
4.0      49
9.0      48
2.0      48
6.0      46
12.0     42
10.0     39
1.0      37
Name: count, dtype: int64
0.0     880
8.0      28
3.0      26
5.0      24
1.0      24
4.0      23
7.0      22
10.0     21
9.0      21
2.0      20
6.0      20
11.0     18
12.0     18
Name: count, dtype: int64
subject #[[3]]
exercise #[[1]]
0.0     339
2.0      54
3.0      52
6.0      47
5.0      47
12.0     46
1.0      44
7.0      43
4.0      42
8.0      40
9.0      39
10.0     35
11.0     34
Name:

In [ ]:
# TODO: Ideas to improve model
# Filter out resting position(Tyler's job)
# Integrate class weights somehow to fix imbalances

print("Starting Training")
epochs = 1000
model.train()

for epoch in range(epochs):
    pbar = tqdm(batched_train, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
    for emg, label in pbar:
        emg = emg.to(device)
        label = label.to(device)
        emg = emg.permute(0, 2, 1) # ! Temporary for this session
        optimizer.zero_grad()
        logits = model(emg)
        loss = loss_func(logits, label)
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=loss.item())
        

Starting Training


KeyboardInterrupt: 

In [32]:
# Accuracy:
model.eval()
total_loss = 0
total = 0
correct = 0
with torch.no_grad():
  correct = 0
  for emg, labels in batched_test:
    emg = emg.to(device)
    emg = emg.permute(0, 2, 1)
    labels = labels.to(device)
    logits = model(emg)
    loss = loss_func(logits, labels)

    total_loss += loss.item() * emg.size(0)
    preds = logits.argmax(dim=1)
    correct += (preds == labels).sum().item()
    total += emg.size(0)


print(f"Average Loss: {total_loss/correct}, Accuracy: {correct/total}")

Average Loss: 17.393405074392444, Accuracy: 0.74108492234477


In [28]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 66.9 MB/s eta 0:00:00


In [34]:
from torchmetrics.classification import MulticlassAccuracy

num_classes = 13
per_class_acc = MulticlassAccuracy(
    num_classes=num_classes,
    average=None
).to(device)

model.eval()

with torch.no_grad():
    for emg, label in batched_test:
        emg = emg.to(device).float()
        label = label.to(device)
        emg = emg.permute(0,2,1)
        logits = model(emg)
        preds = logits.argmax(dim=1)
        per_class_acc.update(preds, label)

accs = per_class_acc.compute()


In [35]:
for i, a in enumerate(accs):
    print(f"Class {i}: {a.item():.3f}")

Class 0: 0.771
Class 1: 0.627
Class 2: 0.727
Class 3: 0.539
Class 4: 0.705
Class 5: 0.753
Class 6: 0.732
Class 7: 0.617
Class 8: 0.700
Class 9: 0.471
Class 10: 0.653
Class 11: 0.699
Class 12: 0.652


In [ ]:
# adaptive memory replay

x2_train, y2_train, x2_test, y2_test, class_weights_2 = preprocessing.multi_preprocess(2, "./NinaProData")
